# train1: temporary training notebook (main line on mix v3 + domain texts: seed repeat and 2x budget)

The main line at 21,600 steps on `mix_v3dom_450k.jsonl` (mix v3 + 50K thesis abstracts/titles and tech/gaming forum posts), `main_v3dom_s21600`, got TR-MTEB **68.27** (mix v3: 68.11; SciFact 78.5 → 80.0). It is the best 1k-base result and the first above our own Qwen3-8B measurement (67.96, +0.31), so before it is reported as the main result:
1. `main_v3dom_s21600_seed43`: the same recipe with seed 43 (the seed changes the student's batch order and the heads' initialisation; the mix and the caches are the same). Seed noise so far was ~0.1 (5400-step pair 67.16 / 67.25).
2. `main_v3dom_s43200`: the same recipe at 43,200 steps (twice the budget, linear decay over the longer run). Gains so far halved with each doubling (5400 → 10,800 +0.31, 10,800 → 21,600 +0.22), so a gain of ~+0.1 to +0.2 is expected; each text is seen ~12 times instead of ~6.

The mix, encoding and the finished 21,600-step run are skipped (their files exist). Approximate A100 time: ~3 h for the seed run, ~6 h for the 43,200-step run (0.44 s/step, not measured on this mix); re-running the cells after a disconnect resumes the cut run. The last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# 1) Mix: v3 + the domain pilot's texts (no raw data needed; both inputs are on Drive)
MIX = f'{DRIVE_ROOT}/data/mix'
V3, DOMAIN = f'{MIX}/mix_v3_400k.jsonl', f'{MIX}/mix_domain_250k.jsonl'
V3DOM, V3DOM_Q = f'{MIX}/mix_v3dom_450k.jsonl', f'{MIX}/mix_v3dom_450k_questions.jsonl'
if not (os.path.exists(V3DOM) and os.path.exists(V3DOM_Q)):
    get_ipython().system(f'python -m turkish_distill.domain_mix --base {V3} --extend-from {DOMAIN} '
                         f'--base-questions {MIX}/mix_v3_400k_questions.jsonl --questions-out {V3DOM_Q} --out {V3DOM}')
from turkish_distill.config import load_config
from turkish_distill.data import load_corpus
from turkish_distill.teachers import corpus_fingerprint
_cfg = load_config('configs/mix_v3dom_two_instr8k.yaml')
_fp = corpus_fingerprint(load_corpus(_cfg.data, _cfg.train.seed))
assert _fp == '47380f3004a8b2eb', f'unexpected mix fingerprint {_fp}'
print('mix ok:', V3DOM, _fp)

# 2) Teacher encoding (skips finished shards; the 39 prefix shards are copied, not encoded)
get_ipython().system(f'python -m turkish_distill.encode_teachers --config configs/stage1_two_bge_m3_modernbert_tr_embed.yaml '
                     f'--override data.corpus_path={V3DOM} --seed-from-corpus {V3}')
get_ipython().system('python -m turkish_distill.encode_teachers --config configs/mix_v3dom_two_instr8k.yaml --only modernbert_tr_embed_instr')

# 3) Runs of this notebook. (name, config, extra overrides). A later override wins.
BASE = []
RUNS = [
    ('main_v3dom_s21600', 'configs/mix_v3dom_two_instr8k.yaml', []),                         # done (68.27): skipped
    ('main_v3dom_s21600_seed43', 'configs/mix_v3dom_two_instr8k.yaml', ['train.seed=43']),  # seed repeat
    ('main_v3dom_s43200', 'configs/mix_v3dom_two_instr8k.yaml', ['train.max_steps=43200']), # 2x budget
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()